In [ ]:
#!pip install simpletransformers transformers==4.40.2

In [ ]:

# Load the required packages

# Dataframes
import pandas as pd, numpy as np

# Regular expressions
import re

# Timestamp / time measurment
import time

# PyTorch: enable GPU access
import torch

# Simpletransformers classifier
# from simpletransformers.classification import ClassificationModel, ClassificationArgs

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
cd /content/drive/MyDrive/manifestos

/content/drive/MyDrive/manifestos


## Load data

In [ ]:
# load full data - update file as appropriate
full = pd.read_csv('combined_post_hoc_unitised.csv')
full.shape

/tmp/ipython-input-4069235983.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  full = pd.read_csv('2026_collection/combined_post_hoc_unitised_12_02_2026.csv') # ('full_25Ju_annot_corrections.csv')


(271899, 14)

In [ ]:
full.columns

Index(['text', 'cmp_code', 'eu_code', 'pos', 'manifesto_id', 'party', 'date',
       'language', 'annotations', 'translation_en', 'sentence_num', 'country',
       'post_hoc_unit', 'qs_new'],
      dtype='object')

## Set up and run model

In [ ]:
model_name = 'manifesto-project/manifestoberta-xlm-roberta-56policy-topics-sentence-2023-1-1'
model_type = 'xlmroberta'

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("manifesto-project/manifestoberta-xlm-roberta-56policy-topics-sentence-2023-1-1")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: manifesto-project/manifestoberta-xlm-roberta-56policy-topics-sentence-2023-1-1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Use this code if need to batch

%%time

predictions = []
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for i in range(0, len(full), batch_size):
    batch_sentences = full.text[i:i+batch_size].tolist()

    try:
        inputs = tokenizer(batch_sentences,
                          return_tensors="pt",
                          max_length=200,
                          padding="max_length",
                          truncation=True
                          ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_classes = [model.config.id2label[idx] for idx in logits.argmax(dim=1).tolist()]
        predictions.extend(predicted_classes)

    except Exception as e:
        print(f"Error at batch {i}: {e}")
        predictions.extend(['None'] * len(batch_sentences))

full['predictions'] = predictions

CPU times: user 2h 29min 28s, sys: 11.7 s, total: 2h 29min 40s
Wall time: 2h 30min 5s


In [ ]:
# Use this code if not

preds,output = model.predict(full['text'].tolist())
predf = pd.DataFrame(zip(full['qs_new'],preds,output),columns=['qs_new','preds','output'])

In [ ]:
predf.shape

(229341, 4)

## Convert prediction outputs to scores

In [ ]:
# id2label = model.config.id2label
# id2label
id2label = {0: '101 - Foreign Special Relationships: Positive',
 1: '102 - Foreign Special Relationships: Negative',
 2: '103 - Anti-Imperialism',
 3: '104 - Military: Positive',
 4: '105 - Military: Negative',
 5: '106 - Peace',
 6: '107 - Internationalism: Positive',
 7: '108 - European Community/Union: Positive',
 8: '109 - Internationalism: Negative',
 9: '110 - European Community/Union: Negative',
 10: '201 - Freedom and Human Rights',
 11: '202 - Democracy',
 12: '203 - Constitutionalism: Positive',
 13: '204 - Constitutionalism: Negative',
 14: '301 - Federalism',
 15: '302 - Centralisation',
 16: '303 - Governmental and Administrative Efficiency',
 17: '304 - Political Corruption',
 18: '305 - Political Authority',
 19: '401 - Free Market Economy',
 20: '402 - Incentives',
 21: '403 - Market Regulation',
 22: '404 - Economic Planning',
 23: '405 - Corporatism/ Mixed Economy',
 24: '406 - Protectionism: Positive',
 25: '407 - Protectionism: Negative',
 26: '408 - Economic Goals',
 27: '409 - Keynesian Demand Management',
 28: '410 - Economic Growth: Positive',
 29: '411 - Technology and Infrastructure',
 30: '412 - Controlled Economy',
 31: '413 - Nationalisation',
 32: '414 - Economic Orthodoxy',
 33: '415 - Marxist Analysis: Positive',
 34: '416 - Anti-Growth Economy: Positive',
 35: '501 - Environmental Protection: Positive',
 36: '502 - Culture: Positive',
 37: '503 - Equality: Positive',
 38: '504 - Welfare State Expansion',
 39: '505 - Welfare State Limitation',
 40: '506 - Education Expansion',
 41: '507 - Education Limitation',
 42: '601 - National Way of Life: Positive',
 43: '602 - National Way of Life: Negative',
 44: '603 - Traditional Morality: Positive',
 45: '604 - Traditional Morality: Negative',
 46: '605 - Law and Order: Positive',
 47: '606 - Civic Mindedness: Positive',
 48: '607 - Multiculturalism: Positive',
 49: '608 - Multiculturalism: Negative',
 50: '701 - Labour Groups: Positive',
 51: '702 - Labour Groups: Negative',
 52: '703 - Agriculture and Farmers: Positive',
 53: '704 - Middle Class and Professional Groups',
 54: '705 - Underprivileged Minority Groups',
 55: '706 - Non-economic Demographic Groups'}

In [ ]:
real_preds = [id2label[i][:3] for i in predf['preds']]

In [ ]:
len(real_preds)

229341

In [ ]:
predf['cmp_code_clean']=real_preds

In [ ]:
predf['cmp_code_clean']=predf['cmp_code_clean'].astype(float)

In [ ]:
predf.rename(columns={'preds':'cmp_pred'},inplace=True)

## Merge with original data

In [ ]:
orig = pd.read_csv('combined_post_hoc_unitised.csv',header=True,index=False)

In [ ]:
# make dummy to identify the codes we just inferred
predf['cmp_inferred']=1

In [ ]:
dfup = pd.merge(orig,predf,on=['qs_new'],how='left', suffixes=('', '_new'))
dfup.shape, orig.shape

In [ ]:
# # Update the original cmp_code_clean with the new scores where they exist
# dfup['cmp_code_clean'] = dfup['cmp_code_clean_new'].combine_first(dfup['cmp_code_clean'])

# # Drop the temporary column used for merging
# dfup.drop(columns=['cmp_code_clean_new'], inplace=True)
# dfup.shape

In [ ]:
dfup.columns

In [ ]:
# Save
dfup.to_csv('combined_post_hoc_unitised_CMP_codes.csv',header=True,index=False)

In [ ]:
# # How many code 416 - pro sustainability?
# dfup['cmp_code_clean'].value_counts(normalize=True)[416.2]+dfup['cmp_code_clean'].value_counts(normalize=True)[416.1]+dfup['cmp_code_clean'].value_counts(normalize=True)[416.0]

In [ ]:
# # How many code 501 - pro environment?
# dfup['cmp_code_clean'].value_counts(normalize=True)[501.0]

## Recreate manifesto-level data - not necessary at this stage

In [ ]:
# meta = pd.read_csv('MPDataset_MPDS2024a.csv')
# meta = meta[meta['date']>199000]
# meta['manifesto_id']=[str(r['party'])+'_'+str(r['date']) for _,r in meta.iterrows()]
# meta.shape

In [ ]:
# meta = meta[
#     [
#  'manifesto_id',
#  'edate',
#  'partyabbrev',
#  'parfam',
#  'manual',
#  'coderyear',
#  'testresult',
#  'testeditsim',
#  'pervote',
#  'voteest',
#  'presvote',
#  'absseat',
#  'totseats',
#  'progtype',
#  'corpusversion',
#  'total',
#  'peruncod','per416',
#  'per501','rile', 'planeco',
#  'markeco',
#  'welfare',
#  'intpeace',
#  'datasetversion']
# ]

In [ ]:
# man_res = dfup.groupby(['country','language','partyname','manifesto_id','parfam_name']).mean(numeric_only=True)[['party','parfam_fixed','year','date','501_raw',
#                                                                                                    '416_raw','final_broad']].reset_index()


In [ ]:
# merged = pd.merge(meta,man_res,on='manifesto_id')
# merged.shape

In [ ]:
# merged.rename(columns={'per416':'416_meta_perc','per501':'501_meta_perc','parfam':'parfam_meta'},inplace=True)

In [ ]:
# merged['final_broad_perc']=merged['final_broad']*100
# merged['501_meta_dec']=merged['501_meta_perc']/100
# merged['416_meta_dec']=merged['416_meta_perc']/100


In [ ]:
# merged.to_csv('manifesto_level_meta_final.csv',header=True,index=False)